## Seção 4.4.1 - Ultrapassagens dos padrões de qualidade do ar no Brasil

Este notebook reproduz a figura 36 da seção 4.4.1 do Relatório Anual de Qualidade do Ar

LEIA ATENTAMENTE ANTES DE EXECUTAR ESTE  SCRIPT

1. ATUALIZAÇÃO DOS CAMINHOS:

   Este código precisa saber em qual pasta do seu computador o projeto está salvo.
   Como cada computador possui uma estrutura de pastas diferente, você deve 
   atualizar a variável de caminho bruto (quando indicado) para
   apontar para o diretório correto na sua máquina antes de rodar o script.


2. PRESERVAÇÃO DA ESTRUTURA DE PASTAS:

   Este código funciona de forma integrada com as demais pastas, scripts e
   arquivos auxiliares exatamente na ORGANIZAÇÃO fornecida no projeto.

   -> Não mova arquivos ou pastas de lugar.

   -> Não altere o nome dos diretórios ou arquivos.

   Caso a estrutura fornecida seja alterada, as importações e chamadas de dados 
   irão falhar e o código não irá funcionar.


### Figura 36 - Faixas de ultrapassagens dos padrões de qualidade do ar, de acordo com a Conama 506/2024
O mapa interativo permite selecionar um padrão de qualidade(PI-1, PI-2,PI-3,PI-4 E PF), um poluente(MP2.5,MP.10,NO2,SO2,O3 e CO) e um ano de monitoramento para visualizar no mapa se a estação excede ou não o padrão selecionado para aquele ano. 

Permite visualizar as faixas e excendencia em uma escala continua de verde, sem excedência, à veremelho, com excedência maior que 100.

Ao clicar no ponto da estação, é possível visualizar o nome da estação, id da estação, poluente selecionado, padrão selecionado, ano selecionado, quantidade de dados válidos, quantidade de dados excendentes, e o percentual de excedência dessa estação de monitoramento.

> **Pré-requisito:** este mapa depende dos arquivos `{PADRAO}/{POLUENTE}_{ANO}.geojson` em `_static/mapas/violacoes/`, gerados por `scripts/mapaViolacoesGeoJson.py` a partir dos dados hospedados remotamente. A célula abaixo executa esse script automaticamente antes de montar o mapa — não é necessário rodá-lo manualmente pelo terminal.

In [ ]:
# Gera (ou regenera) os GeoJSONs de violações a partir dos dados hospedados remotamente.
# Só precisa ser executado uma vez (ou quando os dados de origem forem atualizados) —
# reexecuções subsequentes apenas sobrescrevem os mesmos arquivos em _static/mapas/violacoes/.
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import scripts.mapaViolacoesGeoJson as mapaViol
import importlib; importlib.reload(mapaViol)

mapaViol.mapa_violacoes_geojson()

In [4]:
import json
from pathlib import Path
from IPython.display import HTML

'''  >>> CONFIGURAÇÃO OBRIGATÓRIA <<<
Ajuste VIOLACOES_DIR para a pasta onde os arquivos {PADRAO}/{POLUENTE}_{ANO}.geojson foram
gerados pela célula anterior (scripts/mapaViolacoesGeoJson.py) antes de executar este
script. Como o código é compartilhado via Git, este caminho varia entre usuários. '''
VIOLACOES_DIR = Path("../_static/mapas/violacoes/")

# Carrega, em Python, todos os GeoJSONs de violações disponíveis localmente e embute os
# dados no HTML — evita depender de fetch() em tempo real, que não funciona quando o
# HTML é aberto como arquivo local (file://) fora de um servidor.
data = {}
if VIOLACOES_DIR.exists():
    for padrao_dir in sorted(VIOLACOES_DIR.iterdir()):
        if padrao_dir.is_dir():
            for geo_file in padrao_dir.glob("*.geojson"):
                chave = f"{padrao_dir.name}/{geo_file.stem}"
                with open(geo_file, encoding="utf-8") as f:
                    data[chave] = json.load(f)

data_json = json.dumps(data, ensure_ascii=False)

html_code = fr"""
<!DOCTYPE html>
<html lang="pt-br">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>


<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>

<style>
body {{
  margin: 0;
  padding: 12px;
  font-family: Arial, sans-serif;
  background: #fff;
}}
#controls {{
  display: flex;
  flex-wrap: wrap;
  gap: 12px;
  align-items: center;
  margin-bottom: 10px;
}}
.control {{
  display: flex;
  align-items: center;
  gap: 8px;
  height: 38px;
}}
.label {{
  font-size: 14px;
  font-weight: 600;
  color: #333;
  white-space: nowrap;
}}
.select {{
  appearance: none;
  padding: 6px 10px;
  border: 1px solid #999;
  border-radius: 6px;
  background: #f9f9f9;
  cursor: pointer;
  font-size: 14px;
}}
#status {{
  font-size: 13px;
  color: crimson;
  margin-left: 8px;
}}
#map {{
  width: 100%;
  height: 640px;
  border: 1px solid #ddd;
  opacity: 0;
  transition: opacity .3s;
}}
.leaflet-control.custom-legend {{
  background: #fff;
  padding: 8px;
  border-radius: 6px;
  box-shadow: 0 1px 4px rgba(0,0,0,.4);
  font-size: 12px;
}}
</style>
</head>

<body>


<div id="controls">
  <div class="control">
    <span class="label">Padrão:</span>
    <select id="selPadrao" class="select">
      <option value="PI-1">PI-1</option>
      <option value="PI-2">PI-2</option>
      <option value="PI-3">PI-3</option>
      <option value="PI-4">PI-4</option>
      <option value="PF" selected>PF</option>
    </select>
  </div>

  <div class="control">
    <span class="label">Poluente:</span>
    <select id="selPol" class="select">
      <option value="MP25" selected>MP2,5</option>
      <option value="MP10">MP10</option>
      <option value="NO2">NO2</option>
      <option value="SO2">SO2</option>
      <option value="O3">O3</option>
      <option value="CO">CO</option>
    </select>
  </div>

  <div class="control">
    <span class="label">Ano:</span>
    <select id="selAno" class="select"></select>
  </div>

  <div class="control" style="flex:1;">
    <span id="status"></span>
  </div>
</div>

<div id="map"></div>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>
// === DADOS EMBUTIDOS (Python -> JS) ===
const DATA = {data_json};


// === Preenche anos dinamicamente ===
const selAno = document.getElementById("selAno");
for (let ano = 2024; ano >= 1998; ano--) {{
  const opt = document.createElement("option");
  opt.value = ano;
  opt.textContent = ano;
  if (ano === 2024) opt.selected = true;
  selAno.appendChild(opt);
}}

// === Poluentes formatados ===
const polFmt = {{
  "MP10": "MP10",
  "MP25": "MP2,5",
  "NO2": "NO2",
  "SO2": "SO2",
  "O3": "O3",
  "CO": "CO"
}};

let map = null, layer = null, legend = null;

// === Inicialização do mapa ===
function ensureMap() {{
  if (map) return;
    map = L.map('map', {{
      minZoom: 3,          // permite um pouco mais de afastamento
      maxZoom: 20,         // permite aproximar bem
      maxBounds: L.latLngBounds([-34, -74], [6, -34]),
      maxBoundsViscosity: 0.7 // impede sair do Brasil mas com leve elasticidade
    }}).setView([-14.2, -51.9], 4.5);


  L.tileLayer("https://{{s}}.basemaps.cartocdn.com/rastertiles/voyager/{{z}}/{{x}}/{{y}}.png?key=cb1_2ge2_1_675289d2b5268d90fee0fdab", {{minZoom:2,maxZoom:20,maxNativeZoom:20,subdomains:"abcd",attribution:'&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'}}).addTo(map);
  map.getContainer().style.opacity = 1;
}}

// === Cores ===
function getColor(v) {{
  v = Number(v);
  if (!isFinite(v)) return "gray";            // sem dado
  if (v === 0) return "rgb(0,180,0)";         // verde → 0 Excedência
  if (v <= 10) return "rgb(200,230,0)";       // amarelo-esverdeado
  if (v <= 20) return "rgb(255,220,0)";       // amarelo
  if (v <= 50) return "rgb(255,160,0)";       // laranja
  if (v <= 100) return "rgb(255,80,0)";       // laranja-escuro
  return "rgb(220,0,0)";                      // vermelho intenso
}}

// === Legenda ===
function addLegend() {{
  if (legend) map.removeControl(legend);
  legend = L.control({{ position: "bottomleft" }});
  legend.onAdd = function() {{
    const div = L.DomUtil.create("div", "leaflet-control custom-legend");
    div.innerHTML =
      "<b>Faixas de Excedência:</b><br>" +
      "<div><span style='background:rgb(0,180,0);width:18px;height:10px;display:inline-block;'></span> 0</div>" +
      "<div><span style='background:rgb(200,230,0);width:18px;height:10px;display:inline-block;'></span> 1–10</div>" +
      "<div><span style='background:rgb(255,220,0);width:18px;height:10px;display:inline-block;'></span> 11–20</div>" +
      "<div><span style='background:rgb(255,160,0);width:18px;height:10px;display:inline-block;'></span> 21–50</div>" +
      "<div><span style='background:rgb(255,80,0);width:18px;height:10px;display:inline-block;'></span> 51–100</div>" +
      "<div><span style='background:rgb(220,0,0);width:18px;height:10px;display:inline-block;'></span> >100</div>";
    return div;
  }};
  legend.addTo(map);
}}


// === Popup ===
function popupHTML(p) {{
  const viol = p.VIOLACOES || p.violacoes || "–";
  const nVal = p.N_VALIDOS || p.n_validos || "–";
  const exc = p.PCT_EXC || p.pct_exc;
  const excStr = (exc == null || isNaN(exc)) ? "inválido" : `${{Number(exc).toFixed(1)}}%`;
  const idMMA = p.ID_MMA_COMPLETO || p.id_mma_completo || "–";
  const idOema = p.ID_OEMA || p.id_oema || "–";
  const pol = polFmt[p.POLUENTE] || p.POLUENTE || "–";
  const pad = p.PADRAO || p.padrao || "–";
  const ano = p.ANO || p.ano || "–";

  return `
    <div style='font-family:Arial;font-size:12px;'>
      <b>${{idMMA}}</b><br>
      <i style='color:#555;'>Estação: ${{idOema}}</i><br>
      Poluente: ${{pol}}<br>
      Padrão: ${{pad}}<br>
      Ano: ${{ano}}<br>
      Dados válidos: ${{nVal}}<br>
      Excedência: ${{viol}}<br>
      Excedência: ${{excStr}}
    </div>`;
}}

// === Atualização ===
function updateMap() {{
  ensureMap();
  const padrao = document.getElementById("selPadrao").value;
  const pol = document.getElementById("selPol").value;
  const ano = document.getElementById("selAno").value;
  const status = document.getElementById("status");

  const chave = `${{padrao}}/${{pol}}_${{ano}}`;
  const gj = DATA[chave];

  if (layer) map.removeLayer(layer);
  if (legend) map.removeControl(legend);

  if (!gj) {{
    status.textContent = `⚠️ Dados indisponíveis para ${{padrao}}/${{pol}}/${{ano}}`;
    return;
  }}

  status.textContent = "";
  layer = L.geoJSON(gj, {{
    pointToLayer: (f, latlng) => {{
      const c = getColor(f.properties?.VIOLACOES);
      return L.circleMarker(latlng, {{
        radius: 6,
        color: c,
        weight: 2,
        opacity: 0.4,
        fill: true,
        fillColor: c,
        fillOpacity: 0.55
      }});
    }},
    onEachFeature: (f, l) => l.bindPopup(popupHTML(f.properties || {{}}))
  }}).addTo(map);
  addLegend();
  const b = layer.getBounds();
  if (b.isValid()) map.fitBounds(b, {{ padding: [20, 20] }});
}}

// === Eventos ===
["selPadrao", "selPol", "selAno"].forEach(id => {{
  document.getElementById(id).addEventListener("change", updateMap);
}});

updateMap();
</script>
</body>
</html>
"""

HTML(html_code)

In [ ]:
# Salva o mapa interativo como um arquivo HTML e abre no navegador
import webbrowser
import os

output_dir = "outputs"
output_path = os.path.join(output_dir, "figura36.html")

os.makedirs(output_dir, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_code)

webbrowser.open(output_path)

True